# 7.15 學會新任務會忘記舊任務嗎？


一模型先回答颜色和形状，再只練加法。兩任務共用參數，新更新可能改到舊任務依賴的數，造成遗忘。要發現它，先在兩任務各測一次，再從同一模型继續、重測同一兩份卷子。

下面A題提供「紅是圓、藍是方」的規則，B題是1+1與2+1。四格None是待測记錄，尚沒有成绩；保存問題之外，真评估還需正解與固定生成規則。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
measurements = {
    "微調前": {"A舊任務": None, "B新任務": None},
    "只訓練B後": {"A舊任務": None, "B新任務": None},
}
a_questions = ["紅色物體是圓。請接續：紅色物體是", "藍色物體是方。請接續：藍色物體是"]
b_questions = ["1+1=?", "2+1=?"]
print("固定A題", a_questions)
print("固定B題", b_questions)
print("待填實測", measurements)

輸入兩份固定評估題清單與四格表。None表示尚未測量，不是0分，也不是模型拒答；這段只是建立可記錄的比較接口，沒有在背後訓練模型，所以不填假成績。A問題明確提供前文規則、B是算術，實際任務還應保存正確答案與評分約定，而不只留問句。

真正流程是先訓練得到基模、在A/B各測一次，保存checkpoint；再從它接續只用B資料更新，保存新checkpoint，再測同一A/B題。訓練操作與狀態保存見 [5.7](https://birdhackor.github.io/tiny-perceptron-vlm/5.7.html)、[7.11](https://birdhackor.github.io/tiny-perceptron-vlm/7.11.html)，不能新建隨機模型冒充「同一模型繼續」。評估題應從訓練例之外留出，別把這幾道固定題也不斷用來更新。

若B提升、A下降，可以說新學習伴隨舊退步；若A原先就不會，不能只看A_after下結论。已有屬性基模改練加法後，舊題從5/10降0/10，新加法仍0/7。這只顯示舊內容退步，沒有支持新能力已經學好。EOS與风格也需和內容分開。

練習給A評估清單新增一題，但不加入B訓練資料。先預測此代碼仍只擴大待測清單、四格仍None，再執行核對。你改變的是觀察範圍，不是模型訓練；真實重測時兩個checkpoint都要用更新後同一清單，不能一方多一道難題。

<details>
<summary>補充：實作約定與原始紀錄</summary>

本課實跑使用的A是[7.12的屬性問答](https://birdhackor.github.io/tiny-perceptron-vlm/7.12.html)，B則是0到7的短加法題。從同一份A基模接續，只用49筆B題更新500次後，A的十題最後檢查由5題答對降至0題。原本應回答`circle`的屬性題，這時甚至會生成`9`並正常結束；舊任務的內容表現確實變差，不只是多加一句禮貌話。

但不能把這份結果寫成「新能力學好了，代價是忘掉舊能力」。B在七題最後檢查仍是0題答對，雖然它的平均代價已從7.05820降至4.57756，訓練側代價也降了。這次看到的是舊任務退步，而新任務還沒有在未教過的加法組合上成功。加法兩個加數對調仍放在同一家族，所以`2+3`教過、`3+2`答對不會被算成獨立新組合。下一節把舊題混回來，再看兩邊完整的成績。

短程式沿用本課工具與原計算語義。安裝、長訓練與重做操作見[訓練配方](https://birdhackor.github.io/tiny-perceptron-vlm/training.html)，不需要先完成長配方才能閱讀這個例子。

</details>
